This section reads from the Delta raw table as a stream, parses the `body_str` JSON payload into explicit columns, keeps the raw `ingested_at` timestamp, and writes the result to `dev.raw.ttc_vehicle_positions_bronze`.

In [0]:
from pyspark.sql import functions as F, types as T

source_table = "dev.raw.ttc_vehicle_positions_raw"
target_table = "dev.bronze.ttc_vehicle_positions_bronze"
checkpoint_path = "/Volumes/dev/bronze/checkpoints/ttc_vehicle_positions_bronze"

vehicle_position_schema = T.StructType([
    T.StructField("operation", T.StringType(), True),
    T.StructField("id", T.StringType(), True),
    T.StructField("data", T.StructType([
        T.StructField("entity_id", T.StringType(), True),
        T.StructField("vehicle_id", T.StringType(), True),
        T.StructField("trip_id", T.StringType(), True),
        T.StructField("schedule_relationship", T.StringType(), True),
        T.StructField("route_id", T.StringType(), True),
        T.StructField("stop_id", T.StringType(), True),
        T.StructField("latitude", T.DoubleType(), True),
        T.StructField("longitude", T.DoubleType(), True),
        T.StructField("bearing", T.DoubleType(), True),
        T.StructField("speed", T.DoubleType(), True),
        T.StructField("current_stop_sequence", T.IntegerType(), True),
        T.StructField("current_status", T.StringType(), True),
        T.StructField("occupancy_status", T.StringType(), True),
        T.StructField("timestamp", T.LongType(), True),
        T.StructField("id", T.StringType(), True)
    ]), True)
])

In [0]:
raw_stream = spark.readStream.table(source_table)

bronze_stream = (
    raw_stream
    .select(
        F.from_json(F.col("body_str"), vehicle_position_schema).alias("payload"),
        F.col("ingested_at"),
    )
    .select(
        F.col("payload.operation").alias("operation"),
        F.col("payload.id").alias("event_id"),
        F.col("payload.data.entity_id").alias("entity_id"),
        F.col("payload.data.vehicle_id").alias("vehicle_id"),
        F.col("payload.data.trip_id").alias("trip_id"),
        F.col("payload.data.schedule_relationship").alias("schedule_relationship"),
        F.col("payload.data.route_id").alias("route_id"),
        F.col("payload.data.stop_id").alias("stop_id"),
        F.col("payload.data.latitude").alias("latitude"),
        F.col("payload.data.longitude").alias("longitude"),
        F.col("payload.data.bearing").alias("bearing"),
        F.col("payload.data.speed").alias("speed"),
        F.col("payload.data.current_stop_sequence").alias("current_stop_sequence"),
        F.col("payload.data.current_status").alias("current_status"),
        F.col("payload.data.occupancy_status").alias("occupancy_status"),
        F.to_timestamp(F.from_unixtime(F.col("payload.data.timestamp"))).alias("event_timestamp"),
        F.col("ingested_at"),
    )
)

In [0]:
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .toTable(target_table)
)